
### Gorgon


### Read data

In [2]:
import re
import pandas as pd
df_tag = pd.read_excel(
    "Gorgon Tag Staging Register.xlsx",
    sheet_name="GOR TAG REGISTRATION FORM",
    skiprows=2,
    engine="openpyxl"
)

In [3]:
print(sorted(df_tag.columns.tolist()))

['AA Status', 'ABS Code', 'Ambient Temperature Nameplate Rating', 'Area Gas Group', 'Area Hazardous Zone Rating', 'Area Temperature Class', 'Cause & Effect Drawing', 'Datasheet', 'Discipline', 'EEHA Flag', 'EEHA ITR Date of Inspection', 'EEHA Remarks', 'Equipment Gas Group', 'Equipment Protection Concept', 'Equipment Sub-Class', 'Equipment Temperature Group', 'Ex Certificate Document Number\nNo registered document req. if IEC', 'Ex Certificate of Conformity Number', 'Ex Certifying Authority\nleave blank if IEC', 'From Tag', 'General Arrangement Drawing', 'IP Rating', 'IS Barrier Manufacturer', 'IS Barrier Model', 'IS Calculation Document Number', 'IS Certificate of Conformity Number', 'IS Loop Diagram Document Number', 'Installation & Operating Manual', 'Latest Update', 'Layout Drawing', 'Manufacturer', 'Model', 'NEW / MODIFIED', 'Other Document Reference', 'P&ID', 'Parent Tag Number', 'Physical Device Flag', 'Project or MOC Number', 'Ready For EDW', 'Requestor', 'Serial Number', 'Sing

#### columns to keep 

In [4]:
columns_to_keep4=["Project or MOC Number",
    "TagNumber",
    "AA Status",
    "Ready For EDW",
    "Latest Update",
    "Requestor",
    "NEW / MODIFIED",
    "Tag Description",
    "Equipment Sub-Class",
    "Unit",
    "ABS Code",
    "Parent Tag Number",
    "Discipline"]
df_tag=df_tag[columns_to_keep4]

### Convert to Neo4j naming convention

In [5]:
import re

def to_camel_case(col: str) -> str:
    col = str(col).strip()
    col = re.sub(r"[\s\-]+", "_", col)
    parts = [p for p in col.split("_") if p]
    if not parts:
        return col
    first = parts[0].lower()
    rest = "".join(p[:1].upper() + p[1:].lower() for p in parts[1:])
    return first + rest

df_tag = df_tag.rename(columns=lambda c: to_camel_case(c))
df_tag = df_tag.rename(columns={
    #"Project or MOC Number": "moc_number",
    "tagnumber": "tagNumber",
   
    "new/Modified": "status",
    
})




### Clean projectOrMocNumber column

In [6]:
import re
import pandas as pd
df_tag["latestUpdate"] = (
    pd.to_datetime(df_tag["latestUpdate"], errors="coerce", utc=True)
      .dt.tz_localize(None)   # remove timezone so no +0000 appears
)
# 4) Simple regex to find MOC IDs
pattern = re.compile(r"MOC\s*[_-]?\s*(\d+)", re.IGNORECASE)

def extract_mocs(x):
    if pd.isna(x):
        return []
    s = str(x)

    hits = pattern.findall(s)

    # If digits only, treat as a MOC number
    if not hits and s.isdigit():
        hits = [s]

    return [f"MOC{h}" for h in hits]

df_tag["extractedMocId"] = df_tag["projectOrMocNumber"].apply(extract_mocs)

# 5) Keep only rows with at least one MOC
df_tag = df_tag[df_tag["extractedMocId"].map(len) > 0].copy()

# 6) Explode — one row per MOC
df_tag= df_tag.explode("extractedMocId").reset_index(drop=True)

#    NOTE: TSV stores text; we serialize datetimes in ISO 8601 with timezone if present.
df_tag.to_csv("Cleaned_tag_registrywith_moc.tsv",
    sep="\t",
    index=False,
    date_format="%Y-%m-%d" ) # e.g., 2025-02-09T14:33:00+0000


### Upload to DB

In [10]:
from neo4j import GraphDatabase
import pandas as pd


URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE")
SOR_NAME = "Gorgon_Tag_Registry"  # Change this dynamically for other SORs
BATCH_SIZE = 1000
DATETIME_COLS = [ "latestUpdate"] 
# ---- READ & CLEAN DATA ----

df = df_tag.copy()
# print(df.head(3))
df.columns = df.columns.astype(str).str.strip()

df = df.applymap(lambda v: v.strip() if isinstance(v, str) else v)

NULL_TOKENS = {"", "-", "--", "---"}
df = df.applymap(
    lambda v: None if (isinstance(v, str) and v.strip() in NULL_TOKENS) else v
)
df = df.where(pd.notna(df), None)


rows = []
for r in df.to_dict(orient="records"):
    if r.get("extractedMocId") and r.get("tagNumber"):
        props = {k: v for k, v in r.items() if k not in ["extractedMocId", "tagNumber"]}
        props["tagNumber"] = r["tagNumber"]
        props["extractedMocId"] = r["extractedMocId"]
        rows.append(props)
        
print(f"Prepared {len(rows)} rows for upload.")

# ---- CYPHER ----

cypher = """

UNWIND $rows AS row
MATCH (m:Moc {mocId: row.extractedMocId})

WITH m, row
WHERE row.tagNumber IS NOT NULL

CREATE (t:Tag {tagNumber: row.tagNumber})
SET 
    // merge row properties into tag
    t += row,

    // tag normalization fields
    t.tagNorm = replace(toLower(t.tagNumber), '-', ''),
    t.tagTokens = [x IN split(t.tagNumber, '-') WHERE x <> '' | toLower(x)],

    // manage tag.sourceFile as a list
    t.sourceFile = CASE
        WHEN t.sourceFile IS NULL THEN ['{SOR_NAME}']
        WHEN NOT '{SOR_NAME}' IN t.sourceFile THEN t.sourceFile + ['{SOR_NAME}']
        ELSE t.sourceFile
    END,

    // manage moc.sourceFile as a list
    m.sourceFile = CASE
        WHEN m.sourceFile IS NULL THEN ['{SOR_NAME}']
        WHEN NOT '{SOR_NAME}' IN m.sourceFile THEN m.sourceFile + ['{SOR_NAME}']
        ELSE m.sourceFile
    END,
    t.siteName = "Gorgon"

MERGE (m)-[:ASSOCIATED_WITH_TAG]->(t);
"""

for col in DATETIME_COLS:
    if col in df.columns:
        # Try to parse many formats; invalid -> NaT
        parsed = pd.to_datetime(
            df[col],
            errors="coerce",
            infer_datetime_format=True,  # flexible parsing
            utc=True                     # normalize to UTC; change to False if you want local
        )
        # Keep real datetimes; NaT -> None (becomes null in Neo4j)
        df[col] = parsed.where(parsed.notna(), None)
    else:
        print(f"Warning: datetime column '{col}' not found in TSV — skipping.")

rows = df.to_dict(orient="records")

# ---- UPLOAD IN BATCHES ----
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

with driver.session(database=DATABASE) as session:
    batch = []
    for i, row in enumerate(rows, start=1):
        batch.append(row)
        if len(batch) >= BATCH_SIZE:
            summary = session.run(cypher, rows=batch).consume().counters
            print(f"Batch {i//BATCH_SIZE}: Nodes created={summary.nodes_created}, Relationships={summary.relationships_created}")
            batch.clear()
    if batch:
        summary = session.run(cypher, rows=batch).consume().counters
        print(f"Final batch: Nodes created={summary.nodes_created}, Relationships={summary.relationships_created}")

driver.close()
print("Upload complete!")

C:\Users\rasma\AppData\Local\Temp\ipykernel_36280\687896803.py:22: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda v: v.strip() if isinstance(v, str) else v)
C:\Users\rasma\AppData\Local\Temp\ipykernel_36280\687896803.py:25: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(


Prepared 8645 rows for upload.


C:\Users\rasma\AppData\Local\Temp\ipykernel_36280\687896803.py:81: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(


Batch 1: Nodes created=660, Relationships=660
Batch 2: Nodes created=889, Relationships=889
Batch 3: Nodes created=876, Relationships=876
Batch 4: Nodes created=117, Relationships=117
Batch 5: Nodes created=120, Relationships=120
Batch 6: Nodes created=639, Relationships=639
Batch 7: Nodes created=506, Relationships=506
Batch 8: Nodes created=769, Relationships=769
Final batch: Nodes created=593, Relationships=593
Upload complete!


# Wheatstone

### Read data

In [16]:
import re
import pandas as pd
df_tag_w = pd.read_excel(
    "Wheatstone Tag Staging Register.xlsx",
    sheet_name="WHS TAG REGISTRATION FORM",
    skiprows=2,
    engine="openpyxl"
)

C:\Users\rasma\AppData\Roaming\Python\Python314\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


#### Columns to keep 

In [17]:
import re
import pandas as pd
df_tag_w = pd.read_excel(
    "Wheatstone Tag Staging Register.xlsx",
    sheet_name="WHS TAG REGISTRATION FORM",
    skiprows=2,
    engine="openpyxl"
)

columns_to_keep4=["ProjectorMOCNumber","Discipline",
    "TagNumber",
    'LatestUpdate',
    'ReadyForEDW?', 'NEW/MODIFIED','TagDescription',"Unit",
    "Requestor",'Location','Facility'
    ]
df_tag_w=df_tag_w[columns_to_keep4]

C:\Users\rasma\AppData\Roaming\Python\Python314\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [18]:
df_tag_w= df_tag_w.rename(columns={
    "ProjectorMOCNumber":"projectorMOCNumber","Discipline":"discipline",
    "TagNumber":"tagNumber",
    'LatestUpdate':"latestUpdate",
    'ReadyForEDW?':'readyForEdw', 'NEW/MODIFIED':"status",'TagDescription':"tagDescription","Unit":"unit",
    "Requestor":"requestor",'Location':"location",'Facility':"facility"
})

### Clean projectOrMocNumber column

In [19]:
import re
import pandas as pd
df_tag_w["latestUpdate"] = (
    pd.to_datetime(df_tag_w["latestUpdate"], errors="coerce", utc=True)
      .dt.tz_localize(None)   # remove timezone so no +0000 appears
)
# 4) Simple regex to find MOC IDs
pattern = re.compile(r"MOC\s*[_-]?\s*(\d+)", re.IGNORECASE)

def extract_mocs(x):
    if pd.isna(x):
        return []
    s = str(x)

    hits = pattern.findall(s)

    # If digits only, treat as a MOC number
    if not hits and s.isdigit():
        hits = [s]

    return [f"MOC{h}" for h in hits]

df_tag_w["extractedMocId"] = df_tag_w["projectorMOCNumber"].apply(extract_mocs)

# 5) Keep only rows with at least one MOC
df_tag_w = df_tag_w[df_tag_w["extractedMocId"].map(len) > 0].copy()

# 6) Explode — one row per MOC
df_tag_w= df_tag_w.explode("extractedMocId").reset_index(drop=True)

#    NOTE: TSV stores text; we serialize datetimes in ISO 8601 with timezone if present.
df_tag_w.to_csv("Cleaned_tag_registrywith_moc.tsv",
    sep="\t",
    index=False,
    date_format="%Y-%m-%d" ) # e.g., 2025-02-09T14:33:00+0000


### Upload to DB

In [24]:
from neo4j import GraphDatabase
import pandas as pd


URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE")

SOR_NAME = "Wheatstone_Tag_Registry"  # Change this dynamically for other SORs
BATCH_SIZE = 1000
DATETIME_COLS = [ "latestUpdate"] 
# ---- READ & CLEAN DATA ----

df = df_tag_w.copy()
# print(df.head(3))
df.columns = df.columns.astype(str).str.strip()

df = df.applymap(lambda v: v.strip() if isinstance(v, str) else v)

NULL_TOKENS = {"", "-", "--", "---"}
df = df.applymap(
    lambda v: None if (isinstance(v, str) and v.strip() in NULL_TOKENS) else v
)
df = df.where(pd.notna(df), None)


rows = []
for r in df.to_dict(orient="records"):
    if r.get("extractedMocId") and r.get("tagNumber"):
        props = {k: v for k, v in r.items() if k not in ["extractedMocId", "tagNumber"]}
        props["tagNumber"] = r["tagNumber"]
        props["extractedMocId"] = r["extractedMocId"]
        rows.append(props)
        
print(f"Prepared {len(rows)} rows for upload.")

# ---- CYPHER ----

cypher = """
UNWIND $rows AS row

MATCH (m:Moc {mocId: row.extractedMocId})

WITH m, row
WHERE row.tagNumber IS NOT NULL

CREATE (t:Tag {tagNumber: row.tagNumber})

SET 
    // merge row properties into tag
    t += row,

    // tag normalization fields
    t.tagNorm = replace(toLower(t.tagNumber), '-', ''),
    t.tagTokens = [x IN split(t.tagNumber, '-') WHERE x <> '' | toLower(x)],

    // manage tag.source_file as a list
    t.sourceFile = CASE
        WHEN t.sourceFile IS NULL THEN ['{SOR_NAME}']
        WHEN NOT '{SOR_NAME}' IN t.sourceFile THEN t.sourceFile + ['{SOR_NAME}']
        ELSE t.sourceFile
    END,

    // manage moc.source_file as a list
    m.sourceFile = CASE
        WHEN m.sourceFile IS NULL THEN ['{SOR_NAME}']
        WHEN NOT '{SOR_NAME}' IN m.sourceFile THEN m.sourceFile + ['{SOR_NAME}']
        ELSE m.sourceFile
    END,
    t.siteName ="Wheatstone"
    

MERGE (m)-[:ASSOCIATED_WITH_TAG  ]->(t)
"""

for col in DATETIME_COLS:
    if col in df.columns:
        # Try to parse many formats; invalid -> NaT
        parsed = pd.to_datetime(
            df[col],
            errors="coerce",
            infer_datetime_format=True,  # flexible parsing
            utc=True                     # normalize to UTC; change to False if you want local
        )
        # Keep real datetimes; NaT -> None (becomes null in Neo4j)
        df[col] = parsed.where(parsed.notna(), None)
    else:
        print(f"Warning: datetime column '{col}' not found in TSV — skipping.")

rows = df.to_dict(orient="records")

# ---- UPLOAD IN BATCHES ----
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

with driver.session(database=DATABASE) as session:
    batch = []
    for i, row in enumerate(rows, start=1):
        batch.append(row)
        if len(batch) >= BATCH_SIZE:
            summary = session.run(cypher, rows=batch).consume().counters
            print(f"Batch {i//BATCH_SIZE}: Nodes created={summary.nodes_created}, Relationships={summary.relationships_created}")
            batch.clear()
    if batch:
        summary = session.run(cypher, rows=batch).consume().counters
        print(f"Final batch: Nodes created={summary.nodes_created}, Relationships={summary.relationships_created}")

driver.close()
print("Upload complete!")

C:\Users\rasma\AppData\Local\Temp\ipykernel_36280\4121241075.py:24: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda v: v.strip() if isinstance(v, str) else v)
C:\Users\rasma\AppData\Local\Temp\ipykernel_36280\4121241075.py:27: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(
C:\Users\rasma\AppData\Local\Temp\ipykernel_36280\4121241075.py:85: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(


Prepared 4676 rows for upload.
Batch 1: Nodes created=105, Relationships=105
Batch 2: Nodes created=510, Relationships=510
Batch 3: Nodes created=462, Relationships=462
Batch 4: Nodes created=109, Relationships=109
Final batch: Nodes created=478, Relationships=478
Upload complete!
